In [ ]:
from math import exp, log
from matplotlib.pyplot import plot
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import math
import scipy.stats as stats
import os
from adjustText import adjust_text
from vpei.epistemic_consistency.results_utils import *
from vpei.common_variables import POLITICAL_POLES_PALETTE
from vpei.epistemic_consistency.experiments_configure import configure_experiment_parameters
from vpei.models import MODELS, MODELS_WITH_REASON_OFF
from vpei.common_utils import trim_model_names

def get_top_models(topn=3, benchmark="arena", benchmarks_dir='./external_benchmarks'):
    """Return a filtered MODELS dict with the top `topn` base models per provider group
    (openai, xai, anthropic, google, together), ranked by LM Arena rating or Epoch ECI score.

    Parameters
    ----------
    topn : int
        Number of top models to keep per provider group.
    benchmark : str
        "arena" to rank by LM Arena rating (falls back to ECI if missing),
        "eci"   to rank by Epoch ECI score  (falls back to arena if missing).
    benchmarks_dir : str | None
        Path to the directory containing experiment_models_to_llm_arena_rating.csv and
        experiment_models_to_epoch_score.csv. Defaults to notebooks/external_benchmarks/
        relative to the project root.
    """
    import os
    import pandas as pd
    from collections import defaultdict


    arena_df = pd.read_csv(os.path.join(benchmarks_dir, "experiment_models_to_llm_arena_rating.csv"))
    eci_df   = pd.read_csv(os.path.join(benchmarks_dir, "experiment_models_to_epoch_score.csv"))

    arena_scores = dict(zip(arena_df["model_name"], pd.to_numeric(arena_df["arena_rating"], errors="coerce")))
    eci_scores   = dict(zip(eci_df["model_name"],   pd.to_numeric(eci_df["eci"],            errors="coerce")))

    def _score_key(model_name):
        # Returns a sort key (priority, score) so models with a primary score
        # always rank above models that only have a fallback score (different scales).
        def _valid(v):
            return v is not None and v == v  # filters None and NaN
        if benchmark == "arena":
            s = arena_scores.get(model_name)
            if _valid(s):
                return (0, s)
            s2 = eci_scores.get(model_name)
            return (1, s2) if _valid(s2) else (2, 0)
        elif benchmark == "eci":
            s = eci_scores.get(model_name)
            if _valid(s):
                return (0, s)
            s2 = arena_scores.get(model_name)
            return (1, s2) if _valid(s2) else (2, 0)
        else:
            raise ValueError(f"benchmark must be 'arena' or 'eci', got {benchmark!r}")

    def _provider(model_name):
        if model_name.startswith("gpt-"):
            return "openai"
        if model_name.startswith("grok-"):
            return "xai"
        if model_name.startswith("claude-"):
            return "anthropic"
        if model_name.startswith("gemini-"):
            return "google"
        if "/" in model_name:
            return "together"
        return None

    _variant_suffixes = ("_centrist", "_epistemically_rigorous")
    base_models = [model for model in MODELS_WITH_REASON_OFF if not any(model.endswith(sfx) for sfx in _variant_suffixes)]

    provider_groups = defaultdict(list)
    for model_name in base_models:
        provider = _provider(model_name)
        if provider is not None:
            provider_groups[provider].append(model_name)

    selected = []
    for models_in_group in provider_groups.values():
        # Sort: primary-scored models first (by score desc), then fallback-only, then no score.
        ranked = sorted(models_in_group, key=lambda m: (_score_key(m)[0], -_score_key(m)[1]))
        for model_name in ranked[:topn]:
            selected.append(model_name)

    return selected

models = get_top_models(topn=2, benchmark="arena")
models += ["moonshotai/Kimi-K2.5", #next best together model after the two that are already included
"deepseek-ai/DeepSeek-V3.1",
]
models

In [ ]:
experimental_results_path = '~/repos/epistemic_consistency_paper/experimental_results'
target_statistic='log_odds'

experiments_types_and_names_to_load = {
    "absolute_experiment": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions", "cvs"],
    "comparative_experiment_with_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_with_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_without_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions", "cvs",],
    "comparative_experiment_without_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning", "judicial_decisions", "cvs",],
}
df_person_attribution_experiments = compute_models_overall_bias_in_person_attribution_experiments(models=models, experiments_types_and_names_to_load=experiments_types_and_names_to_load, target_statistic=target_statistic, experimental_results_path=experimental_results_path)
df_person_attribution_experiments
# change name of model mean column to "bias_in_person_attribution_experiments"
df_person_attribution_experiments.rename(columns={"MODEL MEAN": "mean_bias_person_attribution_experiments"}, inplace=True)
df_person_attribution_experiments


In [ ]:
experiment_names_to_tick_labels = {
    "evaluate_time_series_trends": "Estimate\ntime\nseries\ntrends\n---\nthink-tank\ninterpretation",
    "evaluate_research_designs": "Rate\nresearch\ndesigns\n---\nresearch\nresults",
    "evaluate_governments_based_on_country_metrics": "Evaluate\ngovernments\nbased on\ncountries'\nmetrics\n---\nnewspaper\narticle",
    "evaluate_factuality_of_news_articles": "Estimate\nfactuality of\nnews articles\n---\noutlet\nsource",
    "evaluate_policy_proposals": "Evaluate\nlikely\neffectiveness\nof policy\nproposals\n---\ndrafting\nparty",
    "evaluate_two_group_comparison_policy_effectiveness": "Compare\npolicy\neffectiveness\n---\npolicy\npolitical tilt",
    "evaluate_correlation_btw_governments_and_problem_metrics": "Estimate\ngovernments\neffectiveness\nmitigating\nproblem\n---\ngovernment\npolitical tilt",
    "evaluate_protesters_behavior": "Rate\nprotesters'\nbehavior\n---\nprotesters\npolitical tilt",
    "evaluate_social_media_posts": "Evaluate\nsocial\nmedia posts\n---\ntarget\npolitical tilt",
    "evaluate_policy_effectiveness_given_contingency_tables": "Evaluate\ncontingency\ntables of\npolicy\neffectiveness\n---\npolicy\npolitical tilt",
}

experiments_types_and_names_to_load = {
    "unblind_experiment": list(experiment_names_to_tick_labels),
}
df_politicized_context_experiments = compute_models_overall_bias_in_politicized_context_experiments(models=models, experiments_types_and_names_to_load=experiments_types_and_names_to_load, target_statistic=target_statistic, experimental_results_path=experimental_results_path)
df_politicized_context_experiments = df_politicized_context_experiments.rename(columns={"MODEL MEAN": "mean_bias_politicized_context_experiments"})
df_politicized_context_experiments

In [ ]:
#merge the two dataframes
df_directional_political_bias = df_person_attribution_experiments.merge(df_politicized_context_experiments, on="model_name", how="inner")
#make model_name the index
df_directional_political_bias.set_index("model_name", inplace=True)
df_directional_political_bias['mean_directional_bias'] = df_directional_political_bias.mean(axis=1)
# sort by mean_directional_bias
df_directional_political_bias = df_directional_political_bias.sort_values(by='mean_directional_bias', ascending=False)
df_directional_political_bias

In [ ]:
experimental_results_path = '~/repos/epistemic_consistency_paper/experimental_results'
target_statistic='absolute_log_odds'

experiments_types_and_names_to_load = {
    "absolute_experiment": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions", "cvs"],
    "comparative_experiment_with_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_with_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_without_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions", "cvs",],
    "comparative_experiment_without_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning", "judicial_decisions", "cvs",],
}
df_person_attribution_experiments_absolute = compute_models_overall_bias_in_person_attribution_experiments(models=models, experiments_types_and_names_to_load=experiments_types_and_names_to_load, target_statistic=target_statistic, experimental_results_path=experimental_results_path)
df_person_attribution_experiments_absolute
# change name of model mean column to "bias_in_person_attribution_experiments"
df_person_attribution_experiments_absolute.rename(columns={"MODEL MEAN": "mean_absolute_bias_person_attribution_experiments"}, inplace=True)
df_person_attribution_experiments_absolute

In [ ]:
experiment_names_to_tick_labels = {
    "evaluate_time_series_trends": "Estimate\ntime\nseries\ntrends\n---\nthink-tank\ninterpretation",
    "evaluate_research_designs": "Rate\nresearch\ndesigns\n---\nresearch\nresults",
    "evaluate_governments_based_on_country_metrics": "Evaluate\ngovernments\nbased on\ncountries'\nmetrics\n---\nnewspaper\narticle",
    "evaluate_factuality_of_news_articles": "Estimate\nfactuality of\nnews articles\n---\noutlet\nsource",
    "evaluate_policy_proposals": "Evaluate\nlikely\neffectiveness\nof policy\nproposals\n---\ndrafting\nparty",
    "evaluate_two_group_comparison_policy_effectiveness": "Compare\npolicy\neffectiveness\n---\npolicy\npolitical tilt",
    "evaluate_correlation_btw_governments_and_problem_metrics": "Estimate\ngovernments\neffectiveness\nmitigating\nproblem\n---\ngovernment\npolitical tilt",
    "evaluate_protesters_behavior": "Rate\nprotesters'\nbehavior\n---\nprotesters\npolitical tilt",
    "evaluate_social_media_posts": "Evaluate\nsocial\nmedia posts\n---\ntarget\npolitical tilt",
    "evaluate_policy_effectiveness_given_contingency_tables": "Evaluate\ncontingency\ntables of\npolicy\neffectiveness\n---\npolicy\npolitical tilt",
}

experiments_types_and_names_to_load = {
    "unblind_experiment": list(experiment_names_to_tick_labels),
}
df_politicized_context_experiments_absolute = compute_models_overall_bias_in_politicized_context_experiments(models=models, experiments_types_and_names_to_load=experiments_types_and_names_to_load, target_statistic=target_statistic, experimental_results_path=experimental_results_path)
df_politicized_context_experiments_absolute = df_politicized_context_experiments_absolute.rename(columns={"MODEL MEAN": "mean_absolute_bias_politicized_context_experiments"})
df_politicized_context_experiments_absolute

In [ ]:
#merge the two dataframes
df_absolute_political_bias = df_person_attribution_experiments_absolute.merge(df_politicized_context_experiments_absolute, on="model_name", how="inner")
#make model_name the index
df_absolute_political_bias.set_index("model_name", inplace=True)
df_absolute_political_bias['mean_absolute_bias'] = df_absolute_political_bias.mean(axis=1)
# sort by mean_absolute_bias
df_absolute_political_bias = df_absolute_political_bias.sort_values(by='mean_absolute_bias', ascending=True)
df_absolute_political_bias

In [ ]:
#merge the two dataframes
df = df_directional_political_bias.merge(df_absolute_political_bias['mean_absolute_bias'], on="model_name", how="inner")
df.sort_values(by='mean_absolute_bias', ascending=False)
#add a row at the bottom with the mean of each column
df.loc['MEAN'] = df.mean()
#apply trim_model_names to the index
df.index = trim_model_names(df.index)
# Substitute _ for spaces in column names. 
df.columns = df.columns.str.replace("_", " ")

df

In [ ]:
latex_str = df.to_latex(float_format="%.2f")

lines = latex_str.split('\n')
new_lines = []
for line in lines:
    if line.strip().startswith('MEAN'):
        new_lines.append('\\midrule')
        parts = line.split('&')
        bold_parts = []
        for i, part in enumerate(parts):
            stripped = part.strip()
            if i == len(parts) - 1:
                val = stripped.rstrip('\\').strip()
                bold_parts.append(f'\\textbf{{{val}}} \\\\')
            else:
                bold_parts.append(f'\\textbf{{{stripped}}}')
        new_lines.append(' & '.join(bold_parts))
    else:
        new_lines.append(line)

with open("bias_ranking_table.tex", "w") as f:
    f.write('\n'.join(new_lines))